El documento presente sirve de guia para extraer los datos necesarios para la creacion del dataset que nos servirá para la realización del proyecto no.3

Los datos seran comentarios provenientes de videos de youtube, videos de resumenes de las diferentes ediciones del Clasico Tapatio, el clasico mas añejo y con mas historia del futbol mexicano.



In [1]:
pip install youtube-comment-downloader pandas

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.7 MB 4.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.6/10.7 MB 4.7 MB/s eta 0:00:02
   ------ --------------------------------- 1.8/10.7 MB 4.5 MB/s eta 0:00:02
   --------- ------------------------------ 2.6/10.7 MB 3.8 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/10.7 MB 4.1 MB/s eta 0:00:02
   ------------------ --------------------- 5.0/10.7 MB 4.3 MB/s eta 0:00:02
   ---------------------- ----------------- 6.0/10.7 MB 4.5 MB/s eta 0:00:02
   ------------------------ --------------- 6.6/10.7 MB 4.6 MB/s eta 0:00:01
   ---------------------------- ----------- 7.6/10.7 MB 4.3 MB/s eta 0:00:01
   ------------------------------- -------- 8.4/10.7 MB 4.4 MB/s eta 0:00:01
   ---------------------------------- ----- 9.2/10.7 MB 4.2 MB/s eta 0:00:01
   ------------------------------------- -- 10.0/10.7 MB 4.3 MB/s eta 0:00:01
   --


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from itertools import islice
import pandas as pd
from youtube_comment_downloader import SORT_BY_POPULAR, YoutubeCommentDownloader

downloader = YoutubeCommentDownloader()

# Reemplaza con URLs de resúmenes de partidos Chivas vs Atlas en YouTube
urls_clasico = [
    'https://www.youtube.com/watch?v=FTKGRQPDoEs', #2-3 atlas
    'https://www.youtube.com/watch?v=S-sbvU4oags', #4-1 chivas
    'https://www.youtube.com/watch?v=E8-zmPFJl_0', #4-1 chivas
    'https://www.youtube.com/watch?v=BUBmqpB0zFg', #1-1 
    'https://www.youtube.com/watch?v=bgYFaX5VTw8' # 1-2 atlas
]

comentarios_data = []

for url in urls_clasico:
  comentarios = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)
  # Extraemos hasta 800 por video para asegurar >1000 tras limpiar basura
  for comment in islice(comentarios, 800):
    comentarios_data.append({
        'texto': comment['text'],
        'likes': comment['votes'],
        'fecha': comment['time'],
    })

df = pd.DataFrame(comentarios_data)
# Filtrar comentarios muy cortos (menos de 4 palabras) que no sirven para POS/NER
df = df[df['texto'].str.split().str.len() >= 4].reset_index(drop=True)

print(f'Total recolectado: {len(df)} comentarios.')
df.to_csv('dataset_clasico_tapatio_raw.csv', index=False)

Total recolectado: 2218 comentarios.


In [4]:
import re
import numpy as np

# 1. Cargar datos crudos
df = pd.read_csv('dataset_clasico_tapatio_raw.csv')


# 2. Identificar el bloque de video y el ganador implícito por índice
def asignar_contexto_video(idx):
  if idx < 633:
    return 'V1_Victoria_Atlas_2a', 'Atlas'
  elif idx < 1195:
    return 'V2_Victoria_Chivas_1a', 'Chivas'
  elif idx < 1899:
    return 'V3_Victoria_Chivas_3a', 'Chivas'
  elif idx < 1961:
    return 'V4_Empate_4sem', 'Empate'
  else:
    return 'V5_Victoria_Atlas_4a', 'Atlas'


contexto = [asignar_contexto_video(i) for i in range(len(df))]
df['video_id'] = [c[0] for c in contexto]
df['ganador_video'] = [c[1] for c in contexto]

# 3. Diccionarios de expresiones regulares (Jerga + Jugadores + Emojis)
patrones_pro_chivas = [
    # Apoyo directo a Chivas
    r'\b(arriba\s+(las\s+|mis\s+)?chivas|vamos\s+(las\s+|mis\s+)?chivas|mis\s+chivas|soy\s+(de\s+)?chivas?|le\s+voy\s+a\s+(las\s+)?chivas)\b',
    r'\b(chivaherman[oa]s?|rebaño|sagrado|puro\s+chivas|estoeschivas|super\s+chivas|grande\s+(las\s+)?chivas|grande\s+el\s+guadalajara|hivas\s+mandan|chivas\s+mandan)\b',
    # Burlas / Ataques hacia Atlas
    r'\b(ratlas|ratlistas?|caclas|aclas|mieartlas|atlillas|margaritas|equipillo|amargos?|1000\s+derrotas|12\s+t[ií]tulos|tetracampeonato)\b',
    r'\b(hijo\s+eterno\s+de\s+chivas|eterno\s+hijo\s+del\s+guadalajara|chivas\s+es\s+su\s+eterno\s+pap[aá]|su\s+padre\s+las\s+chivas|atlas\s+no\s+trae\s+nada)\b',
    # Figuras de Chivas elogiadas
    r'\b(hormiga|hormigol|armando\s+gonz[aá]lez|otaku\s+del\s+gol|haland\s+mexicano|piojo|piojito|alvarado|mar[ií]n|nene\s+beltr[aá]n|pauno|paunovic|paunovich|bofo|brizuela|cone|tala\s+rangel|milito|sep[uú]lveda|sepu)\b',
    # Emojis Pro-Chivas (antes de limpiarlos)
    r'🐐|🐏|🇫🇷|🔴⚪|⚪🔴|❤️🤍|🤍❤️',
]

patrones_pro_atlas = [
    # Apoyo directo al Atlas
    r'\b(arriba\s+(el\s+|mi\s+)?atlas|vamos\s+(mi\s+|el\s+)?atlas|mi\s+atlas|soy\s+(de\s+|del\s+)?atlas|soy\s+atlista|le\s+voy\s+al\s+atlas|like\s+atlas|atlass+)\b',
    r'\b(rojinegr[oa]s?|siempre\s+fiel|la\s+fiel|academia|a\s+lo\s+atlas|mil\s+veces\s+arriba|grande\s+el\s+atlas|guadalajara\s+es\s+de\s+atlas|gdl\s+es\s+del\s+atlas|es\s+de\s+atlas|ciudad\s+de\s+la\s+furia|orgullo\s+es\s+ser\s+del\s+atlas|esto\s+es\s+atlas)\b',
    # Burlas / Ataques hacia Chivas
    r'\b(chillaherman[oa]s?|chilla\s*herman[oa]s?|chillashermamos|chillas|chibas|chi🅱️as|chigay|chivar|birrias|cabras|borregas|gonzalos?|chichatronco|chichurrito)\b',
    r'\b(inodoro\s+gigante|chiquero|clientes\s+del\s+jeremy|de\s+hijos\s+traemos\s+a\s+las\s+chivas|tra[ií]amos\s+de\s+hijos|chivas\s+ya\s+no\s+gana\s+nada|vives?\s+en\s+el\s+pasado|viven?\s+del\s+pasado|siglo\s+pasado|cavern[ií]cola|cadeneta|amauri)\b',
    # Figuras y logros del Atlas
    r'\b(jeremy|m[aá]rquez|camilo\s+vargas|quiñones|furch|bicampe[oó]n|bicampeones|bicampeonato|campe[oó]n\s+atlas|zague|zaguito)\b',
    # Emojis Pro-Atlas (antes de limpiarlos)
    r'🔴⚫|⚫🔴|♥️🖤|❤🖤|🖤❤️|🖤♥️|🦊|🅱️|♠️♥️|♥️♠️',
]


# 4. Función de pre-etiquetado híbrido (Léxico + Emojis + Contexto del Video)
def pre_etiquetar(row):
  raw = str(row['texto']).lower()
  ganador = row['ganador_video']

  score_c = sum(len(re.findall(p, raw)) * 2 for p in patrones_pro_chivas)
  score_a = sum(len(re.findall(p, raw)) * 2 for p in patrones_pro_atlas)

  # Desambiguación por contexto (ej. burlarse de jugadores rivales o quejarse del arbitraje)
  if ganador == 'Chivas':
    if re.search(
        r'\b(buen\s+partido|partidazo|felicidades|ganamos|goleada|baile|humillando|c[aá]tedra|triunfo|victoria|campeonato|lev[aá]ntense|gusto\s+por\s+las\s+chivas)\b',
        raw,
    ):
      score_c += 2
    if re.search(
        r'\b(fuera\s+de\s+lugar|penal|robo|gato\s+ortiz|se\s+emocionan\s+de\s+nada|celebrando\s+un\s+triunfo|con\s+las\s+[aá]guilas\s+no\s+pudieron|equipo\s+mediocre)\b',
        raw,
    ):
      score_a += 2
  elif ganador == 'Atlas':
    if re.search(
        r'\b(buen\s+partido|partidazo|felicidades|triunfazo|juegazo|ganamos|campe[oó]n|victoria|baile|llorar|pañuelos|lagrimas|ardilla|maternidad|comentaristas|chivas\s*tv)\b',
        raw,
    ):
      score_a += 2
    if re.search(
        r'\b(robo|faul|falta|ayudado|comiendo\s+pasto|3\s+t[ií]tulos|dos\s+titulos|remonatada|ganamos\s+2\s+a\s+0|llorando\s+a\s+papa)\b',
        raw,
    ):
      score_c += 2

  if score_c > score_a:
    return 0, 'Pro_Chivas', 'Alta' if (score_c - score_a) >= 2 else 'Contextual'
  elif score_a > score_c:
    return 1, 'Pro_Atlas', 'Alta' if (score_a - score_c) >= 2 else 'Contextual'
  else:
    # Desempate suave usando la polaridad implícita del video si hay opinión futbolística
    if ganador == 'Chivas':
      return -1, 'Indefinido_En_Video_Chivas', 'Baja'
    elif ganador == 'Atlas':
      return -1, 'Indefinido_En_Video_Atlas', 'Baja'
    return -1, 'Neutral_Empate', 'Baja'


etiquetas = df.apply(pre_etiquetar, axis=1)
df['label'] = [e[0] for e in etiquetas]
df['postura'] = [e[1] for e in etiquetas]
df['confianza'] = [e[2] for e in etiquetas]


# 5. Función de limpieza de texto (preservando estructura para POS y NER)
def limpiar_comentario(texto):
  if pd.isna(texto):
    return ''
  t = str(texto)
  # Quitar caracteres invisibles de YouTube
  t = t.replace('\u200b', ' ').replace('\xa0', ' ')
  # Quitar @menciones de usuarios y URLs
  t = re.sub(r'@[\w\-\.]+', ' ', t)
  t = re.sub(r'http\S+|www\.\S+', ' ', t)
  # Quitar timestamps de video (ej. 1:34, 08:21)
  t = re.sub(r'\b\d{1,2}:\d{2}\b', ' ', t)
  # Quitar emojis y símbolos raros, conservando letras en español, números y puntuación básica
  t = re.sub(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ.,!?;:()\"\'-]', ' ', t)
  # Normalizar letras repetidas excesivas (ej. "Gooooooool" -> "Gool", "jajajajaja" -> "jajaja")
  t = re.sub(r'([a-záéíóúüñA-ZÁÉÍÓÚÜÑ])\1{2,}', r'\1\1', t)
  if re.search(r'(ja){3,}', t, flags=re.IGNORECASE):
    t = re.sub(r'(ja){3,}', 'jaja', t, flags=re.IGNORECASE)
  # Colapsar múltiples espacios y saltos de línea
  t = re.sub(r'\s+', ' ', t).strip()
  return t


df['texto_limpio'] = df['texto'].apply(limpiar_comentario)

# 6. Filtrar ruido: comentarios < 4 palabras, duplicados o alfabeto no latino (ej. fila 1837 en árabe)
df['num_palabras'] = df['texto_limpio'].str.split().str.len()
df_limpio = df[
    (df['num_palabras'] >= 4)
    & (~df['texto'].str.contains(r'[\u0600-\u06FF]', na=False))
].copy()
df_limpio = df_limpio.drop_duplicates(subset=['texto_limpio']).reset_index(
    drop=True
)

# 7. Armar el Dataset Balanceado Final (>= 1,000 comentarios, 50% Chivas y 50% Atlas)
# Priorizamos primero los de confianza 'Alta' y 'Contextual', y completamos con los del video ganador si hace falta
chivas_explicitos = df_limpio[df_limpio['label'] == 0]
atlas_explicitos = df_limpio[df_limpio['label'] == 1]

# Si queremos exactamente 600 y 600 (1,200 total):
n_por_clase = 500

if len(atlas_explicitos) < n_por_clase:
  faltantes_atlas = n_por_clase - len(atlas_explicitos)
  extra_atlas = df_limpio[
      df_limpio['postura'] == 'Indefinido_En_Video_Atlas'
  ].head(faltantes_atlas)
  extra_atlas['label'] = 1
  extra_atlas['postura'] = 'Pro_Atlas'
  atlas_final = pd.concat([atlas_explicitos, extra_atlas])
else:
  atlas_final = atlas_explicitos.head(n_por_clase)

chivas_final = chivas_explicitos.head(n_por_clase)

df_balanceado = (
    pd.concat([chivas_final, atlas_final])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Guardar ambos archivos
df_limpio.to_csv(
    'dataset_clasico_tapatio_completo_limpio.csv',
    index=False,
    encoding='utf-8-sig',
)
df_balanceado[
    ['texto_limpio', 'label', 'postura', 'confianza', 'video_id', 'likes']
].to_csv(
    'dataset_clasico_tapatio_balanceado_1200.csv',
    index=False,
    encoding='utf-8-sig',
)

print(f'Total comentarios limpios válidos: {len(df_limpio)}')
print(f'Distribución en dataset_clasico_tapatio_balanceado_1200.csv:')
print(df_balanceado['postura'].value_counts())

Total comentarios limpios válidos: 2144
Distribución en dataset_clasico_tapatio_balanceado_1200.csv:
postura
Pro_Atlas     500
Pro_Chivas    500
Name: count, dtype: int64


Pasamos de 2218 comentarios a 2144 comentarios validos

500 Pro_Atlas

500 Pro_Chivas 

Esto con la estructura que le dimos, deteccion de patrones, utilizacion de emojis a nuestro favor para clasificar.